In [ ]:
#F0484 Melbourne Housing Market - Price
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from .data_analysis import summarize_categorical, summarize_numeric

pd.set_option('display.width', 1000)
pd.set_option('display.max_columns', None)

filename = 'MELBOURNE_HOUSE_PRICES_LESS.csv'
df = pd.read_csv(filename)
categorical_col = df.select_dtypes(include=['object']).columns.tolist()
numeric_col = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [21]:
#split into train, validation and test ds
rng = np.random.RandomState(42)
perm = rng.permutation(len(df))#shuffle indices

train_split_index = int(0.8 * len(df))
validation_split_index = int(0.9 * len(df))

train_df = df.iloc[perm[:train_split_index]].copy() #select all of the indices up to split_ind and map them to real rows
validation_df = df.iloc[perm[train_split_index:validation_split_index]].copy()
test_df = df.iloc[perm[validation_split_index:]].copy()
print(f"Train:\n {train_df.head(5)}\nValidation:\n{validation_df.head(5)}\nTest: \n{test_df.head(5)}")

Train:
                Suburb             Address  Rooms Type      Price Method     SellerG        Date  Postcode             Regionname  Propertycount  Distance                 CouncilArea
11013      Aberfeldie    8/257 Buckley St      2    u   470000.0      S        Brad   7/05/2016      3040   Western Metropolitan           1543       7.5  Moonee Valley City Council
34702        Armadale  5/92 Wattletree Rd      2    u        NaN     SN     Thomson  18/03/2017      3143  Southern Metropolitan           4836       6.3    Stonnington City Council
52485  Altona Meadows          17 Hall Av      3    u   530000.0     SP      Barlow  26/05/2018      3028   Western Metropolitan           7630      15.5    Hobsons Bay City Council
32079      Camberwell       29 Dominic St      4    h  2650000.0     PI          RT  17/02/2018      3124  Southern Metropolitan           8920       7.7     Boroondara City Council
7696        Yallambie          6 Ovata Cl      3    h   815000.0      S  Buckingha

In [ ]:
num_summary = summarize_numeric(df[numeric_col]) #C
print(num_summary)

          column      N  % missing  cardinality      min        Q1    median         Q3         max           mean            std
0          Rooms  63023       0.00           14      1.0       3.0       3.0        4.0        31.0       3.110595       0.957551
1          Price  63023      23.15         3417  85000.0  620000.0  830000.0  1220000.0  11200000.0  997898.241488  593498.919037
2       Postcode  63023       0.00          225   3000.0    3056.0    3107.0     3163.0      3980.0    3125.673897     125.626877
3  Propertycount  63023       0.00          368     39.0    4380.0    6795.0    10412.0     21650.0    7617.728131    4424.423167
4       Distance  63023       0.00          180      0.0       7.0      11.4       16.7        64.1      12.684829       7.592015


In [ ]:
cat_summary = summarize_categorical(df[categorical_col])
print(cat_summary)

        column      N  % missing  cardinality                     mode  mode freq  mode %                 mode 2  mode 2 freq  mode 2 %
0       Suburb  63023        0.0          380                Reservoir       1241    1.97         Bentleigh East          845      1.34
1      Address  63023        0.0        57754          14 Northcote St          7    0.01           5 Charles St            7      0.01
2         Type  63023        0.0            3                        h      45053   71.49                      u        11655     18.49
3       Method  63023        0.0            9                        S      34063   54.05                     PI         9790     15.53
4      SellerG  63023        0.0          476                    Barry       6894   10.94                 Jellis         5499      8.73
5         Date  63023        0.0          112               24/03/2018       1144    1.82             28/10/2017         1119      1.78
6   Regionname  63023        0.0            8   

In [24]:
#1. duplicate rows
print("Duplicate rows in training df:", train_df.duplicated().sum())
print("Duplicate rows in validation df:", validation_df.duplicated().sum())
train_df = train_df.drop_duplicates()
validation_df = validation_df.drop_duplicates()

Duplicate rows in training df: 1
Duplicate rows in validation df: 0


In [25]:
#2. missing target values
print(f"train_df count before: {len(train_df)}")
train_df = train_df.dropna(subset=['Price'])
print(f"train_df count after: {len(train_df)}")

print(f"validation_df count before: {len(validation_df)}")
validation_df = validation_df.dropna(subset=['Price'])
print(f"validation_df count after: {len(validation_df)}")

train_df count before: 50417
train_df count after: 38707
validation_df count before: 6302
validation_df count after: 4870


In [26]:
#3. Outliers
Q1 = train_df['Price'].quantile(0.25)
Q3 = train_df['Price'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = train_df[
    (train_df['Price'] < lower) |
    (train_df['Price'] > upper)
]
print("Lower bound:", lower)
print("Upper bound:", upper)
print("Number of outliers:", len(outliers))

Lower bound: -280000.0
Upper bound: 2120000.0
Number of outliers: 1845
